## Exercises on Policy-Based Methods (Policy Gradients and REINFORCE)

These paper-and-pencil exercises reinforce Chapter 09: the softmax policy parameterisation, the score function $\nabla_\theta\log\pi_\theta(a\mid s)$, the likelihood-ratio (log-derivative) trick, the policy-gradient theorem and the REINFORCE estimator, and the role of a baseline in reducing variance. (The actor–critic material — advantage estimation, $n$-step critics, GAE — is treated separately in the Chapter 10 exercises to avoid overlap.) Notation: $\pi_\theta(a\mid s)$ is the parameterised policy, $J(\theta)$ the objective, $G_t$ the return.

### Exercise 9.1 — The softmax policy and its score function

A two-action softmax policy uses linear preferences $h(s,a)=\theta^\top\phi(s,a)$ with one-hot features $\phi(s,a_1)=(1,0)$, $\phi(s,a_2)=(0,1)$ and parameters $\theta=(1,0)$.

1. Compute the action probabilities $\pi_\theta(a\mid s)$.
2. Derive the score function $\nabla_\theta\log\pi_\theta(a\mid s)$ for the softmax and evaluate it for $a=a_1$.

**Step 1 — Action probabilities.** $\ \pi_\theta(a\mid s)=\dfrac{e^{h(s,a)}}{\sum_b e^{h(s,b)}}$, with $h(s,a_1)=\theta_1=1,\ h(s,a_2)=\theta_2=0$:

$\displaystyle \pi(a_1\mid s)=\frac{e^{1}}{e^{1}+e^{0}} = 0.7311, \qquad \pi(a_2\mid s)=0.2689.$

**Step 2 — Score function.** Taking $\log$ and differentiating the softmax gives the classic "feature minus expected feature":

$\displaystyle \nabla_\theta \log\pi_\theta(a\mid s) = \phi(s,a) - \sum_{b}\pi_\theta(b\mid s)\,\phi(s,b) = \phi(s,a) - \mathbb{E}_{b\sim\pi}[\phi(s,b)].$

Here $\mathbb{E}_\pi[\phi] = 0.7311(1,0)+0.2689(0,1) = (0.7311, 0.2689)$, so

$\displaystyle \nabla_\theta \log\pi_\theta(a_1\mid s) = (1,0) - (0.7311, 0.2689) = (0.2689, -0.2689).$

**Step 3 — Reading it.** The positive first component means increasing $\theta_1$ raises $\pi(a_1)$; the negative second component means it lowers $\pi(a_2)$. The score points in the direction of parameter space that makes the *taken* action more likely.

**Key concept**

For a softmax policy the score is the **feature of the chosen action minus the policy-averaged feature**. Policy-gradient methods move $\theta$ along this direction, weighted by how good the outcome was.

### Exercise 9.2 — The expected score is zero (why baselines are free)

Prove that the expected score under the policy vanishes, $\ \mathbb{E}_{a\sim\pi_\theta}[\nabla_\theta\log\pi_\theta(a\mid s)] = 0$, and verify it numerically for the policy of Exercise 9.1.

**Step 1 — General proof.** Using $\pi\,\nabla\log\pi = \nabla\pi$ (the log-derivative identity) and $\sum_a\pi_\theta(a\mid s)=1$:

$\displaystyle \mathbb{E}_{a\sim\pi}\big[\nabla_\theta\log\pi_\theta(a\mid s)\big] = \sum_a \pi_\theta(a\mid s)\,\nabla_\theta\log\pi_\theta(a\mid s) = \sum_a \nabla_\theta \pi_\theta(a\mid s) = \nabla_\theta \sum_a \pi_\theta(a\mid s) = \nabla_\theta 1 = 0.$

**Step 2 — Numerical check** (Exercise 9.1 policy). The scores are $\nabla\log\pi(a_1)=(0.2689, -0.2689)$ and $\nabla\log\pi(a_2)=(0,1)-(0.7311, 0.2689)=(-0.7311, 0.7311)$. Then

$\displaystyle \pi(a_1)\,(0.2689, -0.2689) + \pi(a_2)\,(-0.7311, 0.7311) = 0.7311(0.2689, -0.2689) + 0.2689(-0.7311, 0.7311) = (0, 0). \checkmark$

**Step 3 — Consequence.** Because the expected score is zero, subtracting **any** quantity $b(s)$ that does not depend on the action leaves the policy gradient unchanged in expectation:

$\displaystyle \mathbb{E}_a\big[\nabla_\theta\log\pi_\theta(a\mid s)\,b(s)\big] = b(s)\,\mathbb{E}_a\big[\nabla_\theta\log\pi_\theta(a\mid s)\big] = 0.$

**Key concept**

The zero-expected-score identity is the theoretical licence for **baselines**: we may subtract a state-dependent baseline $b(s)$ from the return to reduce variance *without introducing bias*. This is exploited in the next exercise and underpins actor–critic methods.

### Exercise 9.3 — A single REINFORCE update (bandit case)

A one-state (bandit) softmax policy over two actions starts from $\theta=(0,0)$ (so $\pi$ is uniform), with one-hot features as in Exercise 9.1. The agent samples action $a_1$ and receives reward $R=1$. Using the REINFORCE gradient estimate $\hat g = R\,\nabla_\theta\log\pi_\theta(a_1\mid s)$ and learning rate $\alpha=0.2$, perform one update and report the new probability of $a_1$.

**Step 1 — Initial policy.** $\theta=(0,0)\Rightarrow \pi(a_1)=\pi(a_2)=0.5$, so $\mathbb{E}_\pi[\phi]=(0.5,0.5)$.

**Step 2 — Gradient estimate.** Score of the taken action $a_1$: $\ \nabla_\theta\log\pi(a_1)=\phi(a_1)-\mathbb{E}_\pi[\phi]=(1,0)-(0.5,0.5)=(0.5,-0.5).$ With $R=1$:

$\displaystyle \hat g = R\,\nabla_\theta\log\pi(a_1) = (0.5,-0.5).$

**Step 3 — Update.** $\ \theta \leftarrow \theta + \alpha\,\hat g = (0,0) + 0.2(0.5,-0.5) = (0.1, -0.1).$

**Step 4 — New probability.** $\ \pi(a_1) = \dfrac{e^{0.1}}{e^{0.1}+e^{-0.1}} = 0.5498$, up from $0.5$.

Because the reward was positive, the update **increased** the probability of the sampled action.

**Key concept**

REINFORCE performs *likelihood-ratio* gradient ascent: it pushes up the log-probability of actions that led to high return and pushes down those that led to low return. With a positive reward and a uniform start, $\pi(a_1)$ rises.

### Exercise 9.4 — REINFORCE on a two-step episode

An agent uses a **tabular softmax** policy: for each state, two actions with preference parameters, initialised so that every action has probability $0.5$. It runs one episode, $\gamma=1$:

$\displaystyle s_0 \xrightarrow{a_0,\ R_1=0} s_1 \xrightarrow{a_1,\ R_2=1} \text{terminal}.$

Using the **causality** form of the policy gradient (each action weighted by the return *from that step onward*) and $\alpha=0.1$, compute the updates to the preferences of the taken actions and the resulting $\pi(a_0\mid s_0)$.

**Step 1 — Causal returns.** $\ G_0 = R_1+R_2 = 0+1 = 1$ (weights $a_0$), $\quad G_1 = R_2 = 1$ (weights $a_1$).

**Step 2 — Scores for a tabular 2-action softmax.** For the taken action, $\nabla_{\theta(s,a)}\log\pi(a\mid s)=1-\pi(a\mid s)$; for the other action it is $-\pi(\cdot\mid s)$. With every $\pi=0.5$:

$\displaystyle \nabla\log\pi(a_0\mid s_0) = 1-0.5 = 0.5, \qquad \nabla\log\pi(a_1\mid s_1) = 1-0.5 = 0.5.$

**Step 3 — Updates** $\theta(s,a)\leftarrow\theta(s,a)+\alpha\,G_t\,\nabla\log\pi$:

$\displaystyle \theta(s_0,a_0) \leftarrow 0 + 0.1\,(1)(0.5) = 0.05, \qquad \theta(s_0,\text{other}) \leftarrow 0 + 0.1\,(1)(-0.5) = -0.05,$

and identically for $s_1$.

**Step 4 — New probability.** $\ \pi(a_0\mid s_0) = \dfrac{e^{0.05}}{e^{0.05}+e^{-0.05}} = 0.525$, up from $0.5$; the taken action in each visited state becomes more likely.

**Key concept**

REINFORCE credits each action with the return that *follows* it (causality: past rewards cannot depend on the current action). Because the whole episode's return here is positive, every action taken is reinforced — which, without a baseline, is exactly the high-variance behaviour the next exercise addresses.

### Exercise 9.5 — A baseline reduces variance (integrative)

A one-state policy has two actions with equal probability $\pi=(0.5,0.5)$; the (scalar) score of $a_1$ is $+0.5$ and of $a_2$ is $-0.5$ (as in Exercise 9.3). The returns are $q(a_1)=11$ and $q(a_2)=9$ (both positive). Consider the single-sample gradient estimator $\hat g = X(a)\,\nabla\log\pi(a)$, where the action is sampled from $\pi$.

1. With $X(a)=q(a)$ (no baseline), compute $\mathbb{E}[\hat g]$ and $\mathrm{Var}[\hat g]$.
2. With $X(a)=A(a)=q(a)-b$ and baseline $b=v=\mathbb{E}_\pi[q]$, recompute both.
3. Interpret.

**Step 1 — No baseline.** The estimator takes two values: pick $a_1$ ($\hat g=11(0.5)=5.5$) or $a_2$ ($\hat g=9(-0.5)=-4.5$), each with prob. $0.5$.

$\displaystyle \mathbb{E}[\hat g] = 0.5(5.5)+0.5(-4.5) = 0.5, \qquad \mathrm{Var}[\hat g] = 0.5(5.5^2+4.5^2) - 0.5^2 = 25.25-0.25 = 25.$

**Step 2 — With baseline** $b=v=0.5(11)+0.5(9)=10$, so advantages $A(a_1)=+1,\ A(a_2)=-1$. Now $\hat g$ takes: $a_1\!:1(0.5)=0.5$, $a_2\!: (-1)(-0.5)=0.5$ — the **same** value in both cases.

$\displaystyle \mathbb{E}[\hat g] = 0.5 \ (\text{unchanged}), \qquad \mathrm{Var}[\hat g] = 0.5(0.5^2+0.5^2)-0.5^2 = 0.$

**Step 3 — Interpretation.** The baseline left the *expected* gradient unchanged ($0.5$, confirming unbiasedness from Exercise 9.2) but collapsed the variance from $25$ to $0$. Intuitively, when all raw returns are positive the estimator keeps pushing every action up and only the *differences* carry signal; subtracting the mean ($v$) centres the returns so better-than-average actions get positive weight and worse-than-average get negative weight.

**Key concept**

Subtracting a baseline (ideally the state value $v(s)$, giving the **advantage** $A(s,a)=q(s,a)-v(s)$) is the single most important variance-reduction technique in policy-gradient methods. It is *free of bias* precisely because the expected score is zero — the bridge from REINFORCE to actor–critic.